In [2]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

# 1. Load Processed Transactional Data
# Adjust file path if your cleaned transactional dataset is named differently
data_path = r'D:\ecomai\data\processed\cleaned_ecommerce_data.csv'

if os.path.exists(data_path):
  df = pd.read_csv(data_path)
  print('✅ Transaction data loaded successfully.')
else:
  print(f'⚠️ File not found at {data_path}. Please check your file name/path.')

# 2. Build User-Item Interaction Matrix (Quantity Purchased)
user_item_matrix = df.pivot_table(
    index='CustomerID',
    columns='Description',
    values='Quantity',
    aggfunc='sum',
    fill_value=0,
)

print(
    f'📊 User-Item Matrix Shape: {user_item_matrix.shape[0]} Users x'
    f' {user_item_matrix.shape[1]} Products'
)
display(user_item_matrix.head())

✅ Transaction data loaded successfully.
📊 User-Item Matrix Shape: 4338 Users x 3877 Products


Description,4 PURPLE FLOCK DINNER CANDLES,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,I LOVE LONDON MINI RUCKSACK,NINE DRAWER OFFICE TIDY,OVAL WALL MIRROR DIAMANTE,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,...,ZINC STAR T-LIGHT HOLDER,ZINC SWEETHEART SOAP DISH,ZINC SWEETHEART WIRE LETTER RACK,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC TOP 2 DOOR WOODEN SHELF,ZINC WILLIE WINKIE CANDLE STICK,ZINC WIRE KITCHEN ORGANISER,ZINC WIRE SWEETHEART LETTER TRAY
CustomerID,,,,,,,,,,,,,,,,,,,,,
12346,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12347,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12348,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12349,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12350,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
# 1. Convert to Sparse Matrix for Memory Efficiency
sparse_user_item = csr_matrix(user_item_matrix.values)

# 2. Compute Item-to-Item Cosine Similarity (Transpose to compare product columns)
item_similarity = cosine_similarity(sparse_user_item.T)

# 3. Store in DataFrame for Fast Product Lookups
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=user_item_matrix.columns,
    columns=user_item_matrix.columns,
)

print('✅ Engine 1 Built: Item-Item Cosine Similarity Matrix initialized.')

✅ Engine 1 Built: Item-Item Cosine Similarity Matrix initialized.


In [8]:
from mlxtend.frequent_patterns import association_rules, fpgrowth
from sklearn.decomposition import TruncatedSVD

# --- Engine 2: Market Basket Analysis (FP-Growth to avoid memory crash) ---
binary_matrix = user_item_matrix.astype(bool)

# Extract frequent itemsets (min 2% support threshold)
frequent_itemsets = fpgrowth(
    binary_matrix, min_support=0.02, use_colnames=True
)

# Derive Association Rules with Lift > 1.2
rules_df = association_rules(
    frequent_itemsets, metric='lift', min_threshold=1.2
)
rules_df = rules_df.sort_values(by='lift', ascending=False)

# Convert frozensets to lists for pickle serialization compatibility
rules_df['antecedents'] = rules_df['antecedents'].apply(list)
rules_df['consequents'] = rules_df['consequents'].apply(list)

print(
    f'✅ Engine 2 Built (FP-Growth): Generated {len(rules_df)} association'
    ' rules.'
)

# --- Engine 3: Personalized Matrix Factorization (Truncated SVD) ---
svd = TruncatedSVD(n_components=20, random_state=42)
user_factors = svd.fit_transform(user_item_matrix)
item_factors = svd.components_

print(
    '✅ Engine 3 Built: SVD matrix factorization trained with 20 latent'
    ' components.'
)

✅ Engine 2 Built (FP-Growth): Generated 100418 association rules.
✅ Engine 3 Built: SVD matrix factorization trained with 20 latent components.


In [13]:
import os
import joblib

models_dir = r"D:\ecomai\models"
os.makedirs(models_dir, exist_ok=True)

# 1. Save Engine 1: Item-to-Item Cosine Similarity
engine1_path = os.path.join(models_dir, "engine1_item_similarity.pkl")
joblib.dump(item_similarity_df, engine1_path)
print(f"✅ Engine 1 (Product Page) saved to: {engine1_path}")

# 2. Save Engine 2: FP-Growth Association Rules
engine2_path = os.path.join(models_dir, "engine2_cart_rules.pkl")
joblib.dump(rules_df, engine2_path)
print(f"✅ Engine 2 (Cart Bundles) saved to: {engine2_path}")

# 3. Save Engine 3: SVD Matrix Factorization Components & Matrix
engine3_path = os.path.join(models_dir, "engine3_personalized_svd.pkl")
engine3_artifacts = {
    "svd_model": svd,
    "user_factors": user_factors,
    "item_factors": item_factors,
    "user_item_matrix": user_item_matrix,
}
joblib.dump(engine3_artifacts, engine3_path)
print(f"✅ Engine 3 (Personalized Feed) saved to: {engine3_path}")

✅ Engine 1 (Product Page) saved to: D:\ecomai\models\engine1_item_similarity.pkl
✅ Engine 2 (Cart Bundles) saved to: D:\ecomai\models\engine2_cart_rules.pkl
✅ Engine 3 (Personalized Feed) saved to: D:\ecomai\models\engine3_personalized_svd.pkl


In [14]:
import warnings
import joblib
import pandas as pd

# Suppress deprecation warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# --- Load Individual Engine Files ---
item_similarity_df = joblib.load(
    r"D:\ecomai\models\engine1_item_similarity.pkl"
)
rules_df = joblib.load(r"D:\ecomai\models\engine2_cart_rules.pkl")
svd_artifacts = joblib.load(r"D:\ecomai\models\engine3_personalized_svd.pkl")


# --- Define Inference Functions ---
def recommend_similar_items(product_name, top_n=5):
  if product_name not in item_similarity_df.columns:
    return f"Product '{product_name}' not found."
  return (
      item_similarity_df[product_name]
      .sort_values(ascending=False)
      .iloc[1 : top_n + 1]
  )


def get_cart_bundles(product_name, top_n=3):
  matching = rules_df[
      rules_df["antecedents"].apply(lambda x: product_name in x)
  ]
  if matching.empty:
    return "No cart cross-sell rules found for this item."
  return matching[["consequents", "confidence", "lift"]].head(top_n)


def recommend_for_user(customer_id, top_n=5):
  matrix = svd_artifacts["user_item_matrix"]
  if customer_id not in matrix.index:
    return f"Customer ID {customer_id} not found."

  u_idx = matrix.index.get_loc(customer_id)
  predicted_ratings = (
      svd_artifacts["user_factors"][u_idx] @ svd_artifacts["item_factors"]
  )
  preds_series = pd.Series(predicted_ratings, index=matrix.columns)

  already_bought = matrix.loc[customer_id]
  preds_series = preds_series[already_bought == 0]

  return preds_series.sort_values(ascending=False).head(top_n)


# --- Run Test Inferences ---
sample_item = item_similarity_df.columns[0]
sample_user = svd_artifacts["user_item_matrix"].index[0]

print(f"--- 🛒 1. Product Page Recommendations for '{sample_item}' ---")
display(recommend_similar_items(sample_item, top_n=3))

print(f"\n--- 🧺 2. Cart Cross-Sells for '{sample_item}' ---")
display(get_cart_bundles(sample_item, top_n=3))

print(f"\n--- 🎯 3. Personalized Feed for Customer #{sample_user} ---")
display(recommend_for_user(sample_user, top_n=3))

--- 🛒 1. Product Page Recommendations for ' 4 PURPLE FLOCK DINNER CANDLES' ---


Description
4 PINK DINNER CANDLE SILVER FLOCK     0.662486
PINK/FLOWER RABBIT EGG WARMER         0.401943
4 BLUE DINNER CANDLES SILVER FLOCK    0.364795
Name:  4 PURPLE FLOCK DINNER CANDLES, dtype: float64


--- 🧺 2. Cart Cross-Sells for ' 4 PURPLE FLOCK DINNER CANDLES' ---


'No cart cross-sell rules found for this item.'


--- 🎯 3. Personalized Feed for Customer #12346 ---


Description
ASSORTED INCENSE PACK              4.687237
YOU'RE CONFUSING ME METAL SIGN     3.770307
WRAP ALPHABET DESIGN               3.719083
dtype: float64